In [12]:
from pathlib import Path
import pandas as pd

# =========================
# CONFIG TESTING DATASET
# =========================

TEST_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw_Testing")

ROOMS = ["Controlled Room", "Uncontrolled Room"]

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

TEST_SUBJECTS = [
    "Kanaya",
    "Naila",
    "Nana",
    "Rega",
    "Zaira",
]

EXPECTED_FILES = [f"{i}.Csv" for i in range(1, 10)]

REQUIRED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Reflectivity", "Tag"]

expected_total = len(ROOMS) * len(ACTIVITIES) * len(TEST_SUBJECTS) * len(EXPECTED_FILES)

print("Testing base directory:", TEST_BASE_DIR)
print("Testing base exists    :", TEST_BASE_DIR.exists())
print("Rooms                  :", ROOMS)
print("Activities             :", ACTIVITIES)
print("Testing subjects        :", TEST_SUBJECTS)
print("Files per subject       :", EXPECTED_FILES)
print("Expected total CSV      :", expected_total)

Testing base directory: /media/spell/Spell-lab/Lidar/A.Raw_Testing
Testing base exists    : True
Rooms                  : ['Controlled Room', 'Uncontrolled Room']
Activities             : ['Bungkuk', 'Duduk', 'Jongkok', 'Jatuh']
Testing subjects        : ['Kanaya', 'Naila', 'Nana', 'Rega', 'Zaira']
Files per subject       : ['1.Csv', '2.Csv', '3.Csv', '4.Csv', '5.Csv', '6.Csv', '7.Csv', '8.Csv', '9.Csv']
Expected total CSV      : 360


In [13]:
# =========================
# AUDIT TESTING FILE STRUCTURE
# =========================

records = []

for room in ROOMS:
    room_dir = TEST_BASE_DIR / room

    for activity in ACTIVITIES:
        activity_dir = room_dir / activity

        for subject in TEST_SUBJECTS:
            subject_dir = activity_dir / subject
            csv_dir = subject_dir / "CSV"

            for filename in EXPECTED_FILES:
                file_path = csv_dir / filename

                records.append({
                    "room": room,
                    "activity": activity,
                    "subject": subject,
                    "trial_file": filename,
                    "room_dir_exists": room_dir.exists(),
                    "activity_dir_exists": activity_dir.exists(),
                    "subject_dir_exists": subject_dir.exists(),
                    "csv_dir_exists": csv_dir.exists(),
                    "file_exists": file_path.exists(),
                    "file_path": str(file_path),
                })

test_audit_df = pd.DataFrame(records)

total_found = int(test_audit_df["file_exists"].sum())
total_missing = expected_total - total_found

print("===== TESTING DATASET STRUCTURE AUDIT =====")
print("Expected total files:", expected_total)
print("Found files         :", total_found)
print("Missing files       :", total_missing)
print("Completion          : {:.2f}%".format(100 * total_found / expected_total))

print("\nMissing files preview:")
display(test_audit_df[test_audit_df["file_exists"] == False].head(30))

print("\nSummary by room:")
display(
    test_audit_df.groupby("room")["file_exists"]
    .agg(expected="count", found="sum")
    .assign(missing=lambda x: x["expected"] - x["found"])
    .reset_index()
)

print("\nSummary by room and activity:")
display(
    test_audit_df.groupby(["room", "activity"])["file_exists"]
    .agg(expected="count", found="sum")
    .assign(missing=lambda x: x["expected"] - x["found"])
    .reset_index()
)

===== TESTING DATASET STRUCTURE AUDIT =====
Expected total files: 360
Found files         : 360
Missing files       : 0
Completion          : 100.00%

Missing files preview:


,room,activity,subject,trial_file,room_dir_exists,activity_dir_exists,subject_dir_exists,csv_dir_exists,file_exists,file_path



Summary by room:


,room,expected,found,missing
0,Controlled Room,180,180,0
1,Uncontrolled Room,180,180,0



Summary by room and activity:


,room,activity,expected,found,missing
0,Controlled Room,Bungkuk,45,45,0
1,Controlled Room,Duduk,45,45,0
2,Controlled Room,Jatuh,45,45,0
3,Controlled Room,Jongkok,45,45,0
4,Uncontrolled Room,Bungkuk,45,45,0
5,Uncontrolled Room,Duduk,45,45,0
6,Uncontrolled Room,Jatuh,45,45,0
7,Uncontrolled Room,Jongkok,45,45,0


In [14]:
# =========================
# AUDIT TESTING READABILITY & COLUMNS
# =========================

read_records = []

existing_files = test_audit_df[test_audit_df["file_exists"] == True].copy()

for _, row in existing_files.iterrows():
    file_path = Path(row["file_path"])

    status = {
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "trial_file": row["trial_file"],
        "file_path": row["file_path"],
        "file_size_bytes": None,
        "is_empty_file": None,
        "can_read": False,
        "num_rows": None,
        "num_cols": None,
        "columns": None,
        "missing_required_columns": None,
        "has_required_columns": False,
        "error": None,
    }

    try:
        file_size = file_path.stat().st_size
        status["file_size_bytes"] = file_size
        status["is_empty_file"] = file_size == 0

        if file_size == 0:
            status["error"] = "Empty file"
        else:
            # Baca sedikit dulu supaya audit cepat
            temp_df = pd.read_csv(file_path, nrows=5)

            status["can_read"] = True
            status["num_cols"] = len(temp_df.columns)
            status["columns"] = list(temp_df.columns)

            missing_cols = [col for col in REQUIRED_COLUMNS if col not in temp_df.columns]
            status["missing_required_columns"] = missing_cols
            status["has_required_columns"] = len(missing_cols) == 0

            # Hitung jumlah baris total tanpa load full dataframe
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                num_lines = sum(1 for _ in f)

            status["num_rows"] = max(num_lines - 1, 0)

    except Exception as e:
        status["error"] = str(e)

    read_records.append(status)

test_read_audit_df = pd.DataFrame(read_records)

problem_read_df = test_read_audit_df[
    (test_read_audit_df["can_read"] == False) |
    (test_read_audit_df["is_empty_file"] == True) |
    (test_read_audit_df["has_required_columns"] == False)
]

print("===== TESTING READABILITY & COLUMN AUDIT =====")
print("Existing files checked       :", len(test_read_audit_df))
print("Readable files               :", int(test_read_audit_df["can_read"].sum()))
print("Unreadable files             :", int((~test_read_audit_df["can_read"]).sum()))
print("Empty files                  :", int(test_read_audit_df["is_empty_file"].sum()))
print("Files with required columns  :", int(test_read_audit_df["has_required_columns"].sum()))
print("Problem files                :", len(problem_read_df))

print("\nUnreadable / empty / missing column preview:")
display(problem_read_df.head(30))

print("\nRow count statistics:")
display(test_read_audit_df["num_rows"].describe())

===== TESTING READABILITY & COLUMN AUDIT =====
Existing files checked       : 360
Readable files               : 360
Unreadable files             : 0
Empty files                  : 0
Files with required columns  : 360
Problem files                : 0

Unreadable / empty / missing column preview:


,room,activity,subject,trial_file,file_path,file_size_bytes,is_empty_file,can_read,num_rows,num_cols,columns,missing_required_columns,has_required_columns,error



Row count statistics:


count    3.600000e+02
mean     1.169728e+06
std      1.460591e+05
min      7.107850e+05
25%      1.097569e+06
50%      1.150849e+06
75%      1.230073e+06
max      1.943713e+06
Name: num_rows, dtype: float64

In [15]:
# =========================
# SUMMARY PER ROOM, ACTIVITY, SUBJECT
# =========================

merged_test_df = test_audit_df.merge(
    test_read_audit_df[
        [
            "room", "activity", "subject", "trial_file",
            "file_size_bytes", "is_empty_file", "can_read",
            "num_rows", "num_cols", "missing_required_columns",
            "has_required_columns", "error"
        ]
    ],
    on=["room", "activity", "subject", "trial_file"],
    how="left"
)

summary_test_subject = (
    merged_test_df
    .groupby(["room", "activity", "subject"])
    .agg(
        expected_files=("trial_file", "count"),
        found_files=("file_exists", "sum"),
        readable_files=("can_read", "sum"),
        empty_files=("is_empty_file", "sum"),
        files_with_required_columns=("has_required_columns", "sum"),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

summary_test_subject["missing_files"] = summary_test_subject["expected_files"] - summary_test_subject["found_files"]
summary_test_subject["completion_percent"] = 100 * summary_test_subject["found_files"] / summary_test_subject["expected_files"]

summary_test_activity = (
    merged_test_df
    .groupby(["room", "activity"])
    .agg(
        expected_files=("trial_file", "count"),
        found_files=("file_exists", "sum"),
        readable_files=("can_read", "sum"),
        empty_files=("is_empty_file", "sum"),
        files_with_required_columns=("has_required_columns", "sum"),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

summary_test_activity["missing_files"] = summary_test_activity["expected_files"] - summary_test_activity["found_files"]
summary_test_activity["completion_percent"] = 100 * summary_test_activity["found_files"] / summary_test_activity["expected_files"]

print("===== SUMMARY BY ROOM & ACTIVITY =====")
display(summary_test_activity)

print("\n===== SUMMARY BY ROOM, ACTIVITY & SUBJECT =====")
display(summary_test_subject)

print("\nIncomplete/problem subject-activity combinations:")
problem_summary_df = summary_test_subject[
    (summary_test_subject["missing_files"] > 0) |
    (summary_test_subject["readable_files"] < summary_test_subject["expected_files"]) |
    (summary_test_subject["files_with_required_columns"] < summary_test_subject["expected_files"])
]

display(problem_summary_df)

===== SUMMARY BY ROOM & ACTIVITY =====


,room,activity,expected_files,found_files,readable_files,empty_files,files_with_required_columns,total_rows,missing_files,completion_percent
0,Controlled Room,Bungkuk,45,45,45,0,45,53145069,0,100.0
1,Controlled Room,Duduk,45,45,45,0,45,53793165,0,100.0
2,Controlled Room,Jatuh,45,45,45,0,45,57667053,0,100.0
3,Controlled Room,Jongkok,45,45,45,0,45,54249069,0,100.0
4,Uncontrolled Room,Bungkuk,45,45,45,0,45,49309389,0,100.0
5,Uncontrolled Room,Duduk,45,45,45,0,45,50117325,0,100.0
6,Uncontrolled Room,Jatuh,45,45,45,0,45,53764461,0,100.0
7,Uncontrolled Room,Jongkok,45,45,45,0,45,49056621,0,100.0



===== SUMMARY BY ROOM, ACTIVITY & SUBJECT =====


,room,activity,subject,expected_files,found_files,readable_files,empty_files,files_with_required_columns,total_rows,missing_files,completion_percent
0,Controlled Room,Bungkuk,Kanaya,9,9,9,0,9,10303689,0,100.0
1,Controlled Room,Bungkuk,Naila,9,9,9,0,9,10389897,0,100.0
2,Controlled Room,Bungkuk,Nana,9,9,9,0,9,10762953,0,100.0
3,Controlled Room,Bungkuk,Rega,9,9,9,0,9,10498089,0,100.0
4,Controlled Room,Bungkuk,Zaira,9,9,9,0,9,11190441,0,100.0
5,Controlled Room,Duduk,Kanaya,9,9,9,0,9,10362633,0,100.0
6,Controlled Room,Duduk,Naila,9,9,9,0,9,10188873,0,100.0
7,Controlled Room,Duduk,Nana,9,9,9,0,9,10335177,0,100.0
8,Controlled Room,Duduk,Rega,9,9,9,0,9,11364393,0,100.0
9,Controlled Room,Duduk,Zaira,9,9,9,0,9,11542089,0,100.0



Incomplete/problem subject-activity combinations:


,room,activity,subject,expected_files,found_files,readable_files,empty_files,files_with_required_columns,total_rows,missing_files,completion_percent


In [16]:
# =========================
# SAVE TESTING AUDIT REPORTS
# =========================

REPORT_DIR = TEST_BASE_DIR / "_audit_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

detail_report_path = REPORT_DIR / "testing_audit_detail_dataset_completeness.csv"
read_report_path = REPORT_DIR / "testing_audit_readability_columns.csv"
summary_subject_path = REPORT_DIR / "testing_audit_summary_by_room_activity_subject.csv"
summary_activity_path = REPORT_DIR / "testing_audit_summary_by_room_activity.csv"
problem_report_path = REPORT_DIR / "testing_audit_missing_or_problem_files.csv"

merged_test_df.to_csv(detail_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
test_read_audit_df.to_csv(read_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_test_subject.to_csv(summary_subject_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_test_activity.to_csv(summary_activity_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
problem_read_df.to_csv(problem_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")

print("===== TESTING AUDIT REPORT SAVED =====")
print("Detail report          :", detail_report_path)
print("Readability report     :", read_report_path)
print("Summary subject report :", summary_subject_path)
print("Summary activity report:", summary_activity_path)
print("Problem report         :", problem_report_path)

print("\nFinal status:")
print("Expected files:", expected_total)
print("Found files   :", int(merged_test_df["file_exists"].sum()))
print("Missing files :", int(expected_total - merged_test_df["file_exists"].sum()))
print("Readable files:", int(merged_test_df["can_read"].sum()))
print("Problem files :", len(problem_read_df))

===== TESTING AUDIT REPORT SAVED =====
Detail report          : /media/spell/Spell-lab/Lidar/A.Raw_Testing/_audit_reports/testing_audit_detail_dataset_completeness.csv
Readability report     : /media/spell/Spell-lab/Lidar/A.Raw_Testing/_audit_reports/testing_audit_readability_columns.csv
Summary subject report : /media/spell/Spell-lab/Lidar/A.Raw_Testing/_audit_reports/testing_audit_summary_by_room_activity_subject.csv
Summary activity report: /media/spell/Spell-lab/Lidar/A.Raw_Testing/_audit_reports/testing_audit_summary_by_room_activity.csv
Problem report         : /media/spell/Spell-lab/Lidar/A.Raw_Testing/_audit_reports/testing_audit_missing_or_problem_files.csv

Final status:
Expected files: 360
Found files   : 360
Missing files : 0
Readable files: 360
Problem files : 0


Harusnya 

Expected files: 360

Found files   : 360

Missing files : 0

Readable files: 360

Problem files : 0